In [1]:
# Load packages
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle

from FDApy import DenseFunctionalData, MultivariateFunctionalData
from FDApy.representation import DenseArgvals, DenseValues
from FDApy.preprocessing import MFPCA
from FDApy.visualization import plot, plot_multivariate

In [2]:
# Load shots
with open('./data/players_shots_density_attempted.pickle', 'rb') as f:
    players_shots_density = pickle.load(f)
with open('./data/players_shots_density_made.pickle', 'rb') as f:
    players_shots_density_made = pickle.load(f)

In [3]:
# Create functional data
argvals = DenseArgvals({
    'input_dim_0': np.linspace(0, 1, 201),
    'input_dim_1': np.linspace(0, 1, 201)
})

values_shots = DenseValues(
    np.stack([pp / np.max(pp) for pp in players_shots_density['DENSITY'].to_numpy()])
)

values_made = DenseValues(
    np.stack([pp / np.max(pp) for pp in players_shots_density_made['DENSITY'].to_numpy()])
)


fdata_shots = DenseFunctionalData(argvals, values_shots)
fdata_made = DenseFunctionalData(argvals, values_made)
fdata = MultivariateFunctionalData([fdata_shots, fdata_made])

In [18]:
# Bootstrap
for i in np.arange(5):
    sample = np.random.randint(fdata.n_obs, size=fdata.n_obs)
    fdata_sample = fdata[sample]

    # Run MFPCA
    mfpca = MFPCA(n_components=20, method='inner-product')
    mfpca.fit(fdata_sample)

    # Compute the scores
    scores = mfpca.transform(method='InnPro')

    # Inverse transform
    fdata_recons = mfpca.inverse_transform(scores)

    # Save MFPCA results
    with open(f"./data/MFPCA_bootstrap{i}.pickle", 'wb') as f:
        pickle.dump(mfpca, f, protocol=pickle.HIGHEST_PROTOCOL)
    # Save scores
    with open(f"./data/scores_bootstrap{i}.pickle", 'wb') as f:
        pickle.dump(scores, f, protocol=pickle.HIGHEST_PROTOCOL)
    # Save reconstruction
    with open(f"./data/MFPCA_reconstruction{i}.pickle", 'wb') as f:
        pickle.dump(fdata_recons, f, protocol=pickle.HIGHEST_PROTOCOL)

/Users/steven/Documents/workspace/Python/FDApy/FDApy/representation/functional_data.py:1180: UserWarning: The estimation of the variance of the noise is not performed for data with dimension larger than 1 and is set to 0.
  warnings.warn(
